In [ ]:
class Individual:
    """
    Represents a single individual (organism) in the simulation.

    Each Individual possesses a set of diploid chromosome pairs, a unique ID,
    and methods to calculate genetic metrics like hybrid index and heterozygosity,
    and analyze recombination patterns.
    """
    def __init__(self, num_chromosomes: int, num_loci_per_chromosome: int):
        """
        Initializes an Individual object.

        Assigns a unique global ID, sets up the expected number of chromosomes
        and loci, and prepares a list to hold the diploid chromosome pairs.

        Args:
            num_chromosomes (int): The number of diploid chromosome pairs this individual will possess.
            num_loci_per_chromosome (int): The total number of loci (genes) on each individual haploid chromosome.
        """
        # Accessing the global counter to assign a unique ID
        global individual_id_counter
        self.id = individual_id_counter # Assigns a globally unique ID for this individual
        individual_id_counter += 1       # Increments the global counter for the next individual

        self.num_chromosomes = num_chromosomes
        self.num_loci_per_chromosome = num_loci_per_chromosome
        self.diploid_chromosome_pairs: List[DiploidChromosomePair] = [] # Stores the actual homologous chromosome pairs

    def get_all_numeric_genotypes(self) -> List[int]:
        """
        Retrieves the numeric genotype code for every locus across all chromosome pairs of the individual.

        It iterates through each diploid chromosome pair, then through each locus on
        both chromatids within that pair, and uses the `genotype_numeric` helper function
        to determine the genotype (0 for A0A0, 2 for A2A2, 1 for A0A2).

        Returns:
            List[int]: A flattened list containing the numeric genotype (0, 1, or 2)
                       for every locus in the individual's genome.
        """
        all_numeric = []
        for pair in self.diploid_chromosome_pairs:
            alleles_chromatid1 = pair.chromatid1.alleles
            alleles_chromatid2 = pair.chromatid2.alleles
            for i in range(self.num_loci_per_chromosome):
                a1 = alleles_chromatid1[i]
                a2 = alleles_chromatid2[i]
                all_numeric.append(genotype_numeric(a1, a2)) # Uses the standalone helper function
        return all_numeric

    def calculate_hybrid_index(self) -> float:
        """
        Calculates the hybrid index for the individual.

        The hybrid index is a measure of an individual's ancestry, representing
        the proportion of alleles derived from one specific ancestral population (e.g., Allele2).
        It is calculated as: (sum of all numeric locus genotypes) / (2 * total number of loci).
        A value of 0.0 indicates pure Allele0 ancestry, 1.0 indicates pure Allele2 ancestry,
        and 0.5 indicates a perfect F1 hybrid.

        Returns:
            float: The calculated hybrid index, a value between 0.0 and 1.0. Returns 0.0 if no genotypes are present.
        """
        genotypes = self.get_all_numeric_genotypes()
        if not genotypes: # Prevents division by zero if the individual has no genetic data
            return 0.0
        total_possible = 2 * len(genotypes) # Max possible sum if all loci were homozygous Allele2 (2*total_loci)
        total_genotype_sum = sum(genotypes) # Sum of actual numeric genotypes (0, 1, or 2)
        return total_genotype_sum / total_possible

    def calculate_heterozygosity(self) -> float:
        """
        Calculates the overall heterozygosity for the individual.

        Heterozygosity is defined as the proportion of loci that are heterozygous (genotype code 1).
        It provides an indication of genetic diversity within the individual.

        Returns:
            float: The calculated heterozygosity, a value between 0.0 and 1.0. Returns 0.0 if no genotypes are present.
        """
        genotypes = self.get_all_numeric_genotypes()
        if not genotypes: # Prevents division by zero if the individual has no genetic data
            return 0.0
        # Counts how many loci are heterozygous (represented by the numeric code 1)
        return genotypes.count(HETEROZYGOTE) / len(genotypes) # Using the HETEROZYGOTE constant

    def get_chromatid_block_data(self):
        """
        Analyzes each individual chromatid within the individual's diploid chromosome pairs
        to identify contiguous blocks of identical ancestry (alleles) and the recombination
        junctions (breakpoints) between them.

        This method prepares detailed data about the "ancestry blocks" on each chromatid,
        which are crucial for understanding recombination events.

        Returns:
            List[Dict[str, Any]]: A list of dictionaries, where each dictionary represents
            the block analysis for one chromatid. Each dictionary includes:
            - 'individual_id': The unique ID of the individual.
            - 'diploid_chr_id': The 1-based index of the diploid chromosome pair.
            - 'chromatid_in_pair': A label ('A' or 'B') for the specific chromatid within the pair.
            - 'total_junctions': The count of recombination junctions on this chromatid.
            - 'block_lengths': A list of lengths of the contiguous allele blocks.
            - 'block_alleles': A list of the allele types (0 or 2) for each block.
        """
        all_chromatid_data = []
        chromatid_labels = ['A', 'B'] # Labels to distinguish the two chromatids in a diploid pair

        for chr_idx, diploid_pair in enumerate(self.diploid_chromosome_pairs):
            chromatids_in_pair = [diploid_pair.chromatid1, diploid_pair.chromatid2]

            for i, chromatid in enumerate(chromatids_in_pair):
                alleles = chromatid.alleles
                # Calls the private helper method `_analyse_single_chromatid` to do the actual analysis
                junctions, lengths, allele_vals = self._analyse_single_chromatid(alleles)
                all_chromatid_data.append({
                    'individual_id': self.id,
                    'diploid_chr_id': chr_idx + 1,        # 1-based index for the chromosome
                    'chromatid_in_pair': chromatid_labels[i], # 'A' or 'B' (e.g., paternal/maternal origin)
                    'total_junctions': junctions,
                    'block_lengths': lengths,
                    'block_alleles': allele_vals
                })
        return all_chromatid_data

    def _analyse_single_chromatid(self, alleles: List[int]) -> Tuple[int, List[int], List[int]]:
        """
        (Private Helper Method) Analyzes a single chromatid's allele sequence to identify
        contiguous blocks of identical alleles and count the recombination junctions (breakpoints).

        This method is designed to be called internally by `get_chromatid_block_data`.
        It leverages `itertools.groupby` for efficient identification of allele blocks.

        Args:
            alleles (List[int]): The list of numeric alleles (0 or 2) for a single haploid chromatid.

        Returns:
            Tuple[int, List[int], List[int]]: A tuple containing:
                - junctions (int): The total number of recombination breakpoints detected.
                                   (Number of blocks - 1).
                - block_lengths (List[int]): A list where each element is the length (number of loci)
                                             of a contiguous block of identical alleles.
                - block_alleles (List[int]): A list where each element is the allele type (0 or 2)
                                             corresponding to the respective block.
        """
        if not alleles: # Handles the edge case of an empty allele list
            return 0, [], []

        block_lengths = []
        block_alleles = []

        # `itertools.groupby` groups consecutive identical elements.
        # For example, [0, 0, 2, 2, 2, 0] would be grouped into (0, [0,0]), (2, [2,2,2]), (0, [0])
        for allele, group in itertools.groupby(alleles):
            block = list(group)
            block_lengths.append(len(block)) # Store the length of this block
            block_alleles.append(allele)     # Store the allele type of this block

        # The number of junctions is always one less than the number of unbroken blocks
        junctions = len(block_lengths) - 1
        return junctions, block_lengths, block_alleles